In [ ]:
import pandas as pd
import time
from groq import Groq
import csv

# Setup Groq client with API key
client = Groq(api_key="") # insert API key

print("Read CSV ...")
df = pd.read_csv("dataset_clean.csv", 
                 sep=",", 
                 encoding="utf-8-sig")


results = []

# Inference
for i, row in df.iterrows():
    try:
        completion = client.chat.completions.create(
            model="meta-llama/llama-4-scout-17b-16e-instruct",
            messages=[
               {
                    "role": "system", 
                    "content": """Du bist ein Experte für österreichisches Steuerrecht. Antworte präzise und nenne Paragraphen.
                    
                    BEFOLGE DIESE REGELN STRENG:
                    1. Antworte ausschließlich auf Deutsch.
                    2. Nenne immer die genauen Paragraphen.
                    3. Bleibe sachlich und objektiv.
                    4. Wenn ein Sachverhalt nicht eindeutig ist, weise kurz auf die rechtliche Unsicherheit hin.
                    5. Fasse dich kurz: Maximal 3-4 Sätze pro Analyse.
                    6. Gib die Antwort in einer einzigen Zeile aus."""
                },
                {"role": "user", "content": str(row['prompt'])}
            ],
            temperature=0.1,
            max_tokens=150
        )
        
        ans = completion.choices[0].message.content.strip().replace("\n", " ")
        results.append(ans)
        
        if (i + 1) % 50 == 0:
            print(f"Completed: {i + 1}/{len(df)}")
        
        time.sleep(2.5) 

    except Exception as e:
        results.append("Error")
        time.sleep(5) 

# Save results
df['result'] = results
df_result = df[['id', 'result']].copy()
df_result.columns = ['id', 'answer']

df_result.to_csv(
    "inference_model_results.csv", 
    index=False,           
    sep=',',               
    encoding='utf-8-sig',
    quoting=csv.QUOTE_ALL
)

print("CSV completed")


Read CSV ...
Completed: 50/643
Completed: 100/643
Completed: 150/643
Completed: 200/643
Completed: 250/643
Completed: 300/643
Completed: 350/643
Completed: 400/643
Completed: 450/643
Completed: 500/643
Completed: 550/643
Completed: 600/643
CSV completed
